In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import tensorflow as tf
from tensorflow.keras import layers

def resize_to_target_length(x, target_steps=60):
    """
    Hàm nội suy không gian đặc trưng để đồng bộ chiều dài chuỗi.
    Sử dụng thuật toán 'nearest' (Gần nhất) để giữ nguyên độ sắc nét của đặc trưng,
    tránh làm mờ các đỉnh sóng stress.
    """
    x_expanded = tf.expand_dims(x, axis=2) 
    
    x_resized = tf.image.resize(x_expanded, size=[target_steps, 1], method='nearest')
    
    return tf.squeeze(x_resized, axis=2) 

def create_ecg_branch(input_shape=(29760, 1), name_prefix='ECG'):
    inputs = layers.Input(shape=input_shape, name=f'{name_prefix}_Input')
    
    x = layers.Masking(mask_value=0., name=f'{name_prefix}_Masking')(inputs)
    
    x = layers.Conv1D(16, kernel_size=5, padding='causal', name=f'{name_prefix}_Conv1')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4, name=f'{name_prefix}_Pool1')(x)

    x = layers.Conv1D(32, kernel_size=5, dilation_rate=2, padding='causal', name=f'{name_prefix}_DilatedConv2')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    x = layers.MaxPooling1D(pool_size=4, name=f'{name_prefix}_Pool2')(x)

    x = layers.Conv1D(64, kernel_size=3, padding='causal', name=f'{name_prefix}_Conv3')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    
    feature_map = x 
    return inputs, feature_map

def create_slow_branch(input_shape, name_prefix):
    inputs = layers.Input(shape=input_shape, name=f'{name_prefix}_Input')
    x = inputs
    
    x = layers.Conv1D(8, kernel_size=3, padding='causal', name=f'{name_prefix}_Conv0')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_Pool0')(x)

    x = layers.Conv1D(32, kernel_size=3, padding='causal', name=f'{name_prefix}_Conv1')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_Pool1')(x)

    x = layers.Conv1D(64, kernel_size=3, padding='causal', name=f'{name_prefix}_Conv2')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2, name=f'{name_prefix}_Pool2')(x)
    
    x = layers.Conv1D(128, kernel_size=3, padding='causal', name=f'{name_prefix}_Conv3')(x)
    x = layers.Activation('relu')(x)
    x = layers.LayerNormalization()(x)
    
    feature_map = x
    return inputs, feature_map

In [19]:
def build_model():
    in_hgsr, feat_hgsr = create_slow_branch((1860, 1), 'HandGSR')
    in_fgsr, feat_fgsr = create_slow_branch((1860, 1), 'FootGSR')
    in_resp, feat_resp = create_slow_branch((1860, 1), 'RESP')
    in_hr,   feat_hr   = create_slow_branch((930, 1),  'HR')
    in_emg,  feat_emg  = create_slow_branch((930, 1),  'EMG')

    TARGET_STEPS = 60
    
    align_layer = layers.Lambda(
        lambda x: resize_to_target_length(x, TARGET_STEPS), 
    )

    feat_hgsr_aligned = align_layer(feat_hgsr)
    feat_fgsr_aligned = align_layer(feat_fgsr)
    feat_resp_aligned = align_layer(feat_resp)
    feat_hr_aligned   = align_layer(feat_hr)
    feat_emg_aligned  = align_layer(feat_emg)

    merged = layers.Concatenate(axis=-1)([
        feat_hgsr_aligned, feat_fgsr_aligned, 
        feat_resp_aligned, feat_hr_aligned, feat_emg_aligned
    ])

    x = layers.Masking(mask_value=0.0)(merged)
    
    x = layers.LSTM(128, return_sequences=True, dropout=0.3)(x)
    x = layers.LSTM(64, dropout=0.3)(x)
    
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid', name='Stress_Prediction')(x)

    model = tf.keras.models.Model(
        inputs=[in_hgsr, in_fgsr, in_resp, in_hr, in_emg], 
        outputs=output,
        name='DL_Model'
    )
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), 
        loss='binary_crossentropy', 
        metrics=['accuracy']
    )
    return model

In [25]:
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def train_multibranch(data_npz_path):
    data = np.load(data_npz_path)
    sensor_order = ['hand_GSR', 'foot_GSR', 'RESP', 'HR', 'EMG']
    
    X_all_branches = []
    for sensor in sensor_order:
        X_all_branches.append(data[sensor])
    
    y_all = data['labels']
    
    indices = np.arange(y_all.shape[0])
    train_idx, test_idx = train_test_split(
        indices, 
        test_size=0.2, 
        random_state=42, 
        stratify=y_all
    )
    
    X_train_list = [branch[train_idx] for branch in X_all_branches]
    X_test_list = [branch[test_idx] for branch in X_all_branches]
    y_train = y_all[train_idx]
    y_test = y_all[test_idx]
    
    print(f"📊 Tổng số mẫu: {len(indices)}")
    print(f"   - Train: {len(train_idx)} mẫu")
    print(f"   - Test:  {len(test_idx)} mẫu")

    model = build_model()
    
    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
    ]

    history = model.fit(
        x=X_train_list,
        y=y_train,
        validation_data=(X_test_list, y_test),
        epochs=80,
        batch_size=16,
        callbacks=callbacks,
        verbose=1
    )
    
    final_train_acc = history.history['accuracy'][-1]
    final_test_acc = history.history['val_accuracy'][-1]
    
    print("\n" + "="*30)
    print(f"Kết quả cuối cùng (80/20 Split):")
    print(f"👉 Train Accuracy: {final_train_acc*100:.2f}%")
    print(f"👉 Test Accuracy:  {final_test_acc*100:.2f}%")
    print(f"👉 Gap:            {abs(final_train_acc - final_test_acc)*100:.2f}%")
    print("="*30)
    
    return history

history = train_multibranch('/content/drive/MyDrive/data/dl_data.npz')

📊 Tổng số mẫu: 961
   - Train: 768 mẫu
   - Test:  193 mẫu
Epoch 1/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 19s 70ms/step - accuracy: 0.5091 - loss: 0.7086 - val_accuracy: 0.5544 - val_loss: 0.6882 - learning_rate: 1.0000e-04
Epoch 2/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.5391 - loss: 0.6927 - val_accuracy: 0.5648 - val_loss: 0.6891 - learning_rate: 1.0000e-04
Epoch 3/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.5443 - loss: 0.6854 - val_accuracy: 0.5907 - val_loss: 0.6796 - learning_rate: 1.0000e-04
Epoch 4/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - accuracy: 0.5807 - loss: 0.6826 - val_accuracy: 0.5751 - val_loss: 0.6799 - learning_rate: 1.0000e-04
Epoch 5/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.5599 - loss: 0.6839 - val_accuracy: 0.5959 - val_loss: 0.6698 - learning_rate: 1.0000e-04
Epoch 6/80
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.5638 - loss: 0.6791 - val_accuracy: 0.5959 - val_loss: 0.6730 - learning_rate: 1.0000e-04
Epoch 7/80